# **Modelo DeepFM (DeepCTR‑torch)**
### Proyecto Hito 2
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

Este cuaderno entrena un modelo DeepFM usando `dataset_sysrec.csv` y, si está disponible, agrega la característica `genre` desde `video_game_reviews_with_userid.csv`. Incluye:

- Split cálido por usuario (cada usuario aparece en train).
- Entrenamiento y métricas de clasificación (AUC, LogLoss, Accuracy, PR‑AUC, F1).
- Recomendación Top‑N de ejemplo para un usuario.
- Métricas de ranking: Precision@10, Recall@10, F1@10, NDCG@10, HitRate@10, MAP@10, Diversity@10 sobre candidatos por usuario excluyendo vistos.


# Índice

>[0- Instalación de Librerías](#0---instalación-de-librerias)

>[1- Carga de datos](#1---carga-de-datos)

>[2 - Definición del modelo](#2---definición-del-modelo)

>[3 - Recomendacion Top-N](#3---recomendacion-top-n)

>[4 - Análisis](#4---análisis)

>[5 - Anexos](#5---anexos)

# 0 - Instalación de Librerias

In [ ]:
# pip install numpy pandas scikit-learn torch deepctr-torch

In [8]:
import os, random
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, average_precision_score, f1_score

from sklearn.model_selection import train_test_split

from deepctr_torch.inputs import SparseFeat, DenseFeat, get_feature_names
from deepctr_torch.models import DeepFM
from deepctr_torch.callbacks import EarlyStopping


seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = 'cpu'

# 1 - Carga de datos

In [ ]:
base_path = '.'
main_csv = os.path.join(base_path, 'video_game_reviews_with_userid_clean.csv')
items_csv = os.path.join(base_path, 'video_game_reviews_with_userid.csv')

df = pd.read_csv(main_csv)
print('dataset_sysrec.csv ->', df.shape)
print('Columns:', list(df.columns))

required_cols = {'user_id', 'item_id', 'rating'}

items = pd.read_csv(items_csv)
keep_cols = ['item_id']
if 'Genre' in items.columns:
    keep_cols.append('Genre')
if 'Price' in items.columns:
    keep_cols.append('Price')
items = items[keep_cols].dropna(subset=['item_id']).drop_duplicates('item_id')
df = df.merge(items, on='item_id', how='left')

if 'Genre' in df.columns:
    df['Genre'] = df['Genre'].fillna('unknown').astype(str)
    use_genre = True

if 'Price' in df.columns:
    s = df['Price']
    s_clean = s.astype(str).str.replace(r'[^0-9.\-]', '', regex=True)
    price_num = pd.to_numeric(s_clean, errors='coerce')
    med = float(np.nanmedian(price_num)) if np.isfinite(np.nanmedian(price_num)) else 0.0
    price_num = price_num.fillna(med).astype(float)
    price_log = np.log1p(price_num)
    # min-max para escalar los precios
    pmin, pmax = float(np.min(price_log)), float(np.max(price_log))
    if np.isfinite(pmin) and np.isfinite(pmax) and pmax > pmin:
        price_scaled = (price_log - pmin) / (pmax - pmin)
    else:
        price_scaled = price_log * 0.0
    df['price'] = price_scaled.astype('float32')
    use_price = True
    print('Items con Price ->', df['price'].notna().sum(), '| use_price=True')
# Vista inicial    
print('Head:')
print(df.head(3))
print('Rating mean:', float(df['rating'].mean()))


dataset_sysrec.csv -> (47774, 3)
Columns: ['user_id', 'item_id', 'rating']
Items con Price -> 47774 | use_price=True
Head:
   user_id  item_id    rating      Genre  Price     price
0      861       12  3.670051  Adventure  41.41  0.664772
1     1295       38  3.862944    Shooter  57.56  0.971821
2     1131       21  2.695431  Adventure  44.93  0.740647
Rating mean: 2.9918100005121406


Se cargan los archivos dataset_sysrec.csv (interacciones) y video_game_reviews_with_userid.csv (catálogo), se conservan las columnas item_id y con Genre y Price, se realiza un merge por item_id. Para Price se limpia el texto, se convierte a numérico, se imputan faltantes con la mediana, se aplica log1p y se escala min–max en la nueva columna price, activando use_price. 

# 2 - Definición del modelo

En **DeepCTR-torch**, el modelo DeepFM está diseñado originalmente para predecir *Click-Through Rate (CTR)*, por lo que la variable objetivo debe ser **binaria**:  
- `0` si el usuario no realizó la acción (no clickeó),  
- `1` si el usuario sí la realizó (clickeó).  

Por lo que como el promedio en este dataset de ratings es cercano a 3. Se tomará como positivo si es mayor a 4.

In [5]:
pos_threshold = 4
if 'label' not in df.columns:
    df['label'] = (df['rating'] >= pos_threshold).astype('int8')

In [ ]:
# Codificación (LabelEncoder) para user, item y genre
user_raw_col, item_raw_col = 'user_id', 'item_id'
le_user, le_item = LabelEncoder(), LabelEncoder()

df['user'] = le_user.fit_transform(df[user_raw_col].astype(str))
df['item'] = le_item.fit_transform(df[item_raw_col].astype(str))

n_users = df['user'].nunique(); n_items = df['item'].nunique()
print(f'n_users={n_users}, n_items={n_items}')


le_genre = LabelEncoder()
df['genre_enc'] = le_genre.fit_transform(df['Genre'].astype(str))
n_genres = df['genre_enc'].nunique()
print('n_genres=', n_genres)


n_users=3000, n_items=40
n_genres= 9


Aca user_id e item_id a índices enteros consecutivos mediante LabelEncoder y Genre a genre_enc. Luego calcula y muestra las cardinalidades n_users, n_items y n_genres, que se usarán como vocabulary_size en las SparseFeat del modelo DeepFM. 

Ahora se divide en train y test:


In [9]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=seed, stratify=df["label"])
val_df, test_df  = train_test_split(val_df, test_size=0.5, random_state=seed, stratify=val_df["label"])

In [ ]:
# Definir columnas de características 
embedding_dim_user_item = 32
embedding_dim_genre = 32  

fixlen_feature_columns = [
    SparseFeat('user', vocabulary_size=n_users, embedding_dim=embedding_dim_user_item),
    SparseFeat('item', vocabulary_size=n_items, embedding_dim=embedding_dim_user_item),
]
if use_genre and 'genre_enc' in df.columns:
    fixlen_feature_columns.append(SparseFeat('genre_enc', vocabulary_size=n_genres, embedding_dim=embedding_dim_genre))

# Dense features 
dense_feature_columns = []
if use_price and 'price' in df.columns:
    dense_feature_columns.append(DenseFeat('price', 1))

linear_feature_columns = fixlen_feature_columns + dense_feature_columns
dnn_feature_columns = fixlen_feature_columns + dense_feature_columns
feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

sparse_feature_names = ['user', 'item'] + (['genre_enc'] if (use_genre and 'genre_enc' in df.columns) else [])
dense_feature_names = []
if use_price and 'price' in df.columns:
    dense_feature_names.append('price')

print('Features:', feature_names)
print('Sparse features:', sparse_feature_names)
print('Dense features:', dense_feature_names)

Features: ['user', 'item', 'genre_enc', 'price']
Sparse features: ['user', 'item', 'genre_enc']
Dense features: ['price']


Aca crea SparseFeat para user, item y gerne con embeddings de 32 dimensiones y DenseFeat para price. A partir de ello arma las listas lineales y del DNN, obtiene los feature_names y separa los campos categóricos (sparse) de los numéricos (dense) para construir los inputs. 



In [11]:
def build_X(df_):
    data = {}
    for name in feature_names:
        if name in df_.columns:
            if name in globals().get('dense_feature_names', []):
                data[name] = df_[name].astype('float32').values
            else:
                data[name] = df_[name].astype('int64').values
    return data

In [12]:
X_train, y_train = build_X(train_df), train_df["label"].values
X_val, y_val = build_X(val_df), val_df["label"].values
X_test, y_test = build_X(test_df), test_df["label"].values
{k: v.shape for k, v in X_train.items()}


{'user': (38219,), 'item': (38219,), 'genre_enc': (38219,), 'price': (38219,)}

Ahora se ejecuta el modelo con Adam y binary_crossentropy y entrena 10 épocas 

In [ ]:
print(pos_threshold)
model = DeepFM(
    linear_feature_columns=linear_feature_columns,
    dnn_feature_columns=dnn_feature_columns,
    task='binary',
    device=device,
    dnn_hidden_units=(256, 128),
    dnn_dropout=0.4,
)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['auc'])

epochs = 10
batch_size = 2048
history = model.fit(X_train, y_train,
                    batch_size=batch_size, epochs=epochs, verbose=2,
                    validation_data=(X_val, y_val) if len(val_df)>0 else None,
                    shuffle=True)

4
cpu
Train on 38219 samples, validate on 4777 samples, 19 steps per epoch
Epoch 1/10
0s - loss:  0.5869 - auc:  0.5032 - val_auc:  0.5073
Epoch 2/10
0s - loss:  0.3669 - auc:  0.5580 - val_auc:  0.5097
Epoch 3/10
0s - loss:  0.3328 - auc:  0.6737 - val_auc:  0.5177
Epoch 4/10
0s - loss:  0.3174 - auc:  0.7250 - val_auc:  0.5195
Epoch 5/10
0s - loss:  0.3058 - auc:  0.7315 - val_auc:  0.5190
Epoch 6/10
0s - loss:  0.2999 - auc:  0.7348 - val_auc:  0.5209
Epoch 7/10
0s - loss:  0.2981 - auc:  0.7354 - val_auc:  0.5213
Epoch 8/10
0s - loss:  0.2971 - auc:  0.7374 - val_auc:  0.5183
Epoch 9/10
0s - loss:  0.2963 - auc:  0.7390 - val_auc:  0.5201
Epoch 10/10
0s - loss:  0.2959 - auc:  0.7385 - val_auc:  0.5195


Se el modelo está aprendiendo en train pero no generaliza en val. Esto se puede deber a que hay señales débiles/escasas y pocos ítems (n_items=40), lo que lleva a fácil sobreajuste. Más adelante se realizará mas en detalle un analisis.

In [ ]:
pred_test = model.predict(X_test, batch_size=2048)
pred_test = np.asarray(pred_test).reshape(-1)
y_test_vec = np.asarray(y_test).reshape(-1)
auc  = roc_auc_score(y_test_vec, pred_test)
ll   = log_loss(y_test_vec, pred_test)
pred_label = (pred_test >= 0.5).astype('int8')
acc  = accuracy_score(y_test_vec, pred_label)
prauc = average_precision_score(y_test_vec, pred_test)
print(f"AUC={auc:.4f} | LogLoss={ll:.4f} | Accuracy={acc:.4f} | PR-AUC={prauc:.4f}")

AUC=0.5244 | LogLoss=0.3979 | Accuracy=0.8945 | PR-AUC=0.1132


# 3 - Recomendacion Top-N

In [ ]:
def recommend_top_n(user_id_raw, N=10):
    # Convertir a str para usar el encoder tal como fue entrenado
    if isinstance(user_id_raw, (int, np.integer)):
        user_id_raw = str(user_id_raw)
    try:
        u_idx = le_user.transform([user_id_raw])[0]
    except Exception as e:
        raise ValueError('user_id desconocido en el encoder') from e

    # Excluir ítems vistos en train y validación
    seen_train = set(train_df.loc[train_df['user'] == u_idx, 'item'].tolist())
    seen_val = set(val_df.loc[val_df['user'] == u_idx, 'item'].tolist()) if 'val_df' in globals() else set()
    seen = seen_train.union(seen_val)

    candidates = [i for i in range(n_items) if i not in seen]
    if not candidates:
        return []

    # Construir batches para predecir
    scores_chunks = []
    batch = 4096
    for start in range(0, len(candidates), batch):
        end = min(start + batch, len(candidates))
        X = {'user': np.full(end-start, u_idx, dtype='int64'),
             'item': np.array(candidates[start:end], dtype='int64')}
        if 'genre_enc' in feature_names:
            if 'item_to_genre' not in globals():
                tmp = df[['item','genre_enc']].dropna() if 'genre_enc' in df.columns else pd.DataFrame({'item':[], 'genre_enc':[]})
                globals()['item_to_genre'] = dict(zip(tmp['item'].astype(int), tmp['genre_enc'].astype(int)))
            ge = np.array([globals()['item_to_genre'].get(i, 0) for i in candidates[start:end]], dtype='int64')
            X['genre_enc'] = ge
        if 'price' in feature_names:
            if 'item_to_price' not in globals():
                tmp_p = df[['item','price']].dropna() if 'price' in df.columns else pd.DataFrame({'item':[], 'price':[]})
                globals()['item_to_price'] = dict(zip(tmp_p['item'].astype(int), tmp_p['price'].astype(float)))
                globals()['price_fallback'] = float(df['price'].median()) if 'price' in df.columns else 0.0
            pr = np.array([globals()['item_to_price'].get(i, globals().get('price_fallback', 0.0)) for i in candidates[start:end]], dtype='float32')
            X['price'] = pr

        X = {name: X[name] for name in feature_names}
        s = model.predict(X, batch_size=len(X['user']))
        s = np.asarray(s).reshape(-1)
        scores_chunks.append(s)
    scores = np.concatenate(scores_chunks) if scores_chunks else np.array([])
    if scores.size == 0:
        return []

    order = np.argsort(scores)[-N:][::-1]
    top_items_idx = np.array(candidates)[order]
    top_scores = scores[order]
    item_raw_ids = le_item.inverse_transform(top_items_idx)
    try:
        item_raw_ids = item_raw_ids.astype(int)
    except Exception:
        pass
    return list(zip(item_raw_ids.tolist(), top_scores.astype(float).tolist()))

# Usuario Aleatorio de test para ejemplo de recomendación
if len(test_df) > 0:
    sample_u_idx = int(test_df['user'].sample(1, random_state=None).iloc[0])
    sample_u_raw = le_user.inverse_transform([sample_u_idx])[0]
    print('Usuario de ejemplo:', sample_u_raw)
    print('Top‑10:', recommend_top_n(sample_u_raw, N=10))

Usuario de ejemplo: 2549
Top‑10: [(24, 0.06192297115921974), (37, 0.06094631552696228), (16, 0.06062660738825798), (13, 0.05803438276052475), (2, 0.056689951568841934), (5, 0.056470174342393875), (19, 0.055921148508787155), (21, 0.05584856867790222), (40, 0.054082754999399185), (29, 0.0524803102016449)]


Aca la idea es generar recomendaciones Top‑N para un usuario usando el modelo DeepFM ya entrenado, excluyendo ítems vistos. En primer lugar se reutiliza los LabelEncoder de user e item para moverse entre IDs y los índices internos del modelo. Usa mapeos item→genre_enc y item→price (cacheados en globals) Se normaliza el user_id de entrada y lo transforma al índice interno, luego se construye el conjunto de candidatos excluyendo ítems vistos por ese usuario en train y val.

Para todos los candidatos, arma batches:
- user: vector constante con u_idx.
- item: índices de ítem candidatos.
- genre_enc y price: se agregan desde los mapeos, price usa mediana como fallback.

X se reordena exactamente como feature_names y se asegura dtype: int64 para sparse, float32 para dense. Predice con model.predict por batch, y concatena scores.
Ordena por score descendente y toma los N mejores, luegp convierte los índices de ítem a item_id originales con le_item.inverse_transform. Finalmente devuelve lista de pares (item_id_original, score).


In [ ]:
# Métricas de ranking @10 sobre candidatos por usuario
import numpy as np
import pandas as pd
from collections import defaultdict

def build_user_seen(dfs, user_col='user', item_col='item'):
    seen = defaultdict(set)
    for d in dfs:
        if d is None or len(d) == 0:
            continue
        for u, i in zip(d[user_col].values, d[item_col].values):
            seen[int(u)].add(int(i))
    return seen


def precision_recall_at_k_ranked(df_user_ranked, k):
    n = len(df_user_ranked)
    denom_p = min(k, n) if n > 0 else 1
    topk = df_user_ranked.head(k)
    hits_k = int(topk['label'].sum())
    total_rel = int(df_user_ranked['label'].sum())
    prec = hits_k / denom_p if denom_p > 0 else np.nan
    rec = hits_k / total_rel if total_rel > 0 else np.nan
    return prec, rec, total_rel


def ndcg_at_k_ranked(df_user_ranked, k):
    topk = df_user_ranked.head(k)
    rel = topk['label'].to_numpy(dtype=float)
    if rel.size == 0:
        return np.nan
    discounts = np.log2(np.arange(2, 2 + rel.size))
    dcg = np.sum(rel / discounts)
    total_rel = int(df_user_ranked['label'].sum())
    ideal_ones = min(k, total_rel)
    if ideal_ones == 0:
        return np.nan
    ideal_rel = np.ones(ideal_ones, dtype=float)
    idcg = np.sum(ideal_rel / np.log2(np.arange(2, 2 + ideal_ones)))
    return dcg / idcg if idcg > 0 else np.nan


def ap_at_k_ranked(df_user_ranked, k):
    topk = df_user_ranked.head(k)
    labels = topk['label'].astype(int).to_numpy()
    total_rel = int(df_user_ranked['label'].sum())
    if labels.size == 0:
        return 0.0 if total_rel == 0 else 0.0
    if total_rel == 0:
        return 0.0
    denom = min(k, total_rel)
    hits = 0
    ap = 0.0
    for i, y in enumerate(labels, start=1):
        if y == 1:
            hits += 1
            ap += hits / i
    return ap / denom if denom > 0 else 0.0


def hit_rate_at_k_ranked(df_user_ranked, k):
    topk = df_user_ranked.head(k)
    return 1.0 if int(topk['label'].sum()) > 0 else 0.0


def macro_pr_f1_ndcg_at_k(ranked_df, k):
    precs, recs, f1s, ndcgs = [], [], [], []
    for _, df_u in ranked_df.groupby('user'):
        p, r, _ = precision_recall_at_k_ranked(df_u, k)
        nd = ndcg_at_k_ranked(df_u, k)
        if not np.isnan(p):
            precs.append(p)
        if not np.isnan(r):
            recs.append(r)
        if (not np.isnan(p)) and (not np.isnan(r)) and (p + r) > 0:
            f1s.append((2 * p * r) / (p + r))
        if not np.isnan(nd):
            ndcgs.append(nd)
    mp = float(np.mean(precs)) if len(precs) > 0 else np.nan
    mr = float(np.mean(recs)) if len(recs) > 0 else np.nan
    mf1 = float(np.mean(f1s)) if len(f1s) > 0 else np.nan
    mndcg = float(np.mean(ndcgs)) if len(ndcgs) > 0 else np.nan
    return mp, mr, mf1, mndcg


all_items = np.array(sorted(df['item'].unique()))
user_seen = build_user_seen([train_df, val_df], user_col='user', item_col='item')

test_pos = test_df[test_df['label'] == 1].groupby('user')['item'].apply(set).to_dict()
users = sorted(test_df['user'].unique())

K = 10
NEG_PER_USER = 500  

pairs = []
rng = np.random.default_rng(42)

item_to_genre = {}
if 'genre_enc' in feature_names and 'genre_enc' in df.columns:
    item_to_genre = dict(zip(df['item'].astype(int), df['genre_enc'].astype(int)))

item_to_price = {}
if 'price' in feature_names and 'price' in df.columns:
    item_to_price = dict(zip(df['item'].astype(int), df['price'].astype(float)))
    price_fallback = float(df['price'].median())

for u in users:
    seen_u = user_seen.get(int(u), set())
    candidates_u = np.setdiff1d(all_items, np.fromiter(seen_u, dtype=int), assume_unique=True)
    pos_u = np.array(sorted(test_pos.get(int(u), set())), dtype=int)
    neg_pool = candidates_u if pos_u.size == 0 else candidates_u[~np.isin(candidates_u, pos_u)]
    if NEG_PER_USER is None or neg_pool.size <= (NEG_PER_USER or 0):
        sampled_neg = neg_pool
    else:
        sampled_idx = rng.choice(neg_pool.size, size=NEG_PER_USER, replace=False)
        sampled_neg = neg_pool[sampled_idx]
    for i in sampled_neg:
        pairs.append((int(u), int(i), 0))
    for i in pos_u:
        pairs.append((int(u), int(i), 1))

rank_df = pd.DataFrame(pairs, columns=['user', 'item', 'label'])

# Predicciones de score
X_rank = {'user': rank_df['user'].astype('int64').values, 'item': rank_df['item'].astype('int64').values}
if 'genre_enc' in feature_names:
    ge = np.array([item_to_genre.get(int(i), 0) for i in rank_df['item'].values], dtype='int64')
    X_rank['genre_enc'] = ge
if 'price' in feature_names:
    pr = np.array([item_to_price.get(int(i), price_fallback) for i in rank_df['item'].values], dtype='float32')
    X_rank['price'] = pr

X_rank = {name: X_rank[name] for name in feature_names}
scores = model.predict(X_rank, batch_size=4096)
rank_df['score'] = np.asarray(scores).reshape(-1)

# Ordenar y calcular métricas 
rank_df = rank_df.sort_values(['user', 'score'], ascending=[True, False]).reset_index(drop=True)
mp, mr, mf1, mndcg = macro_pr_f1_ndcg_at_k(rank_df, K)

hr_list, ap_list, div_list = [], [], []
has_genre = ('genre_enc' in feature_names) and (len(item_to_genre) > 0)
for _, df_u in rank_df.groupby('user'):
    topk = df_u.head(K)
    hr_list.append(1.0 if int(topk['label'].sum()) > 0 else 0.0)
    total_rel = int(df_u['label'].sum())
    if total_rel == 0:
        ap_list.append(0.0)
    else:
        hits = 0
        ap = 0.0
        for i, y in enumerate(topk['label'].astype(int).tolist(), start=1):
            if y == 1:
                hits += 1
                ap += hits / i
        ap_list.append(ap / float(min(K, total_rel)))
    if has_genre:
        genres = [item_to_genre.get(int(i), None) for i in topk['item'].values]
        genres = [g for g in genres if g is not None]
        if len(genres) > 0:
            div_list.append(len(set(genres)) / float(min(K, len(topk))))

mhr = float(np.mean(hr_list)) if len(hr_list) > 0 else np.nan
mmap = float(np.mean(ap_list)) if len(ap_list) > 0 else np.nan
mdiv = float(np.mean(div_list)) if (len(div_list) > 0 and has_genre) else np.nan

metrics_at_10 = pd.DataFrame([
    {
        'K': K,
        'Precision@10': mp,
        'Recall@10': mr,
        'F1@10': mf1,
        'NDCG@10': mndcg,
        'HitRate@10': mhr,
        'MAP@10': mmap,
        'Diversity@10': mdiv,
    }
])
metrics_at_10

,K,Precision@10 (macro),Recall@10 (macro),F1@10 (macro),NDCG@10 (macro),HitRate@10 (macro),MAP@10 (macro),Diversity@10 (unique genres ratio) (macro)
0,10,0.008438,0.401971,0.183998,0.18283,0.082284,0.022512,0.666793


Ahora se evalúo el ranking a nivel usuario con K=10. Para ello, se construye un conjunto de pares (user,item) con positivos reales de test y un muestreo de negativos, predice scores con el modelo y calcula métricas Top‑K agregadas. En primer lugar, se excluye ítems vistos en train/val por usuario (build_user_seen). Para cada usuario en test, se ven los candidatos (todos los items no vistos), positivos (items de test con label=1) y negativos (candidatos sin los positivos; se muestrean hasta NEG_PER_USER (500) por eficiencia). Luego se arma rank_df[user,item,label] con todos esos pares. Despues, se genera X_rank respetando feature_names y dtypes (user, item, genre_enc, price), para asi predecir score, con probabilidad con model.predict y lo añade a rank_df.

Esto lo que hace entonces es, ordenar rank_df por usuario y score desc, y luego calcular por usuario sobre el top‑K:
Precision@10, Recall@10, F1@10 y NDCG@10.
HitRate@10: proporción de usuarios con al menos un acierto en su Top‑10.
MAP@10: promedio macro de Average Precision truncada a K.
Diversity: fracción de géneros únicos en el Top‑K si genre_enc existe.
Agrega los promedios en un DataFrame metrics_at_10 y lo muestra.

# 4 - Análisis


### Los resultados obtenidos son los siguientes:

- Clasificación (test): AUC=0.5244, LogLoss=0.3979, Accuracy=0.8945 , PR-AUC=0.1132 (muy baja).
- Top‑N (ranking macro@10): Precision=0.0084 (0.84%), Recall=0.402, F1=0.184, NDCG=0.183, HitRate=0.082, MAP=0.0225, Diversity≈0.667.
Scores de recomendación muy bajos (~0.05–0.06) y muy juntos entre ítems.

### Qué significan estos números

El valor del AUC cercano a 0.52 indica que el modelo apenas logra una capacidad de separación ligeramente mejor que el azar, lo que sugiere que las variables utilizadas no presentan una correlación significativa con la etiqueta binaria. En otras palabras, el modelo no está capturando patrones relevantes que permitan distinguir correctamente entre las clases positivas y negativas.

Además, el PR-AUC tiende a acercarse a la prevalencia de la clase positiva, lo que en este caso refuerza la idea de que el modelo no está aprendiendo más allá del comportamiento base.

Los valores promedio de las puntuaciones (0.05–0.06) muestran que el modelo asigna bajas probabilidades a la mayoría de los pares (usuario, ítem), lo que refleja una actitud conservadora ante la falta de señales predictivas útiles. En la práctica, esto implica que el sistema cree que casi ningún par será positivo, reduciendo la capacidad de recomendar ítems verdaderamente relevantes.

En cuanto al rendimiento en ranking, la métrica Precision@10 = 0.0084 indica que, en promedio, solo 0.084 ítems del top-10 por usuario son relevantes, lo cual es bajo. El HitRate@10 = 0.082 confirma que apenas un 8.2% de los usuarios recibe al menos una recomendación correcta dentro de sus diez primeras posiciones, mientras que el 91.8% restante no obtiene ningún acierto. Sin embargo, el Recall@10 (macro) alcanza 0.402, lo que puede parecer contradictorio, pero se explica porque este se calcula solo sobre los usuarios que sí poseen al menos un positivo en el conjunto de prueba. Si la mayoría tiene muy pocos positivos, basta un solo acierto para que el recall de ese subconjunto sea alto.

Finalmente, las métricas de ordenamiento, como NDCG y MAP, resultan bajas, indicando que el modelo no logra priorizar correctamente los ítems relevantes en las primeras posiciones del ranking. En contraste, la diversidad promedio de ≈0.667 sugiere que en el top-10 de cada usuario aparecen cerca de 6.7 géneros distintos. Si bien una alta diversidad podría parecer positiva, en este caso refleja que el modelo no tiene preferencias definidas y que el ranking es difuso, lo que refuerza la idea de un comportamiento casi aleatorio.


### Explicación

Los resultados obtenidos con el modelo DeepFM fueron considerablemente bajos, lo que sugiere la presencia de una señal débil en las features utilizadas. En este caso, las variables disponibles *user*, *item*, *genre_enc* y *price* no parecen contener información suficiente para capturar patrones reales de preferencia o comportamiento. Esto se ve agravado por el tamaño reducido del catálogo (n_items ≈ 40) y la limitada o potencialmente ruidosa metadata, lo que dificulta que el modelo aprenda relaciones significativas entre usuarios e ítems.

Además, como el dataset fue generado aleatoriamente, los ratings o etiquetas no reflejan preferencias consistentes ni comportamientos reales de usuarios. En este escenario, los ítems no presentan una estructura de similitud o relevancia, y el modelo carece de una señal útil para distinguir entre interacciones positivas y negativas. Incluso el identificador del ítem (*item_id*) deja de tener valor predictivo, ya que no existe un patrón estable que asocie ciertos ítems con ciertos tipos de usuarios. En consecuencia, el modelo aprende esencialmente ruido y su rendimiento termina siendo apenas superior al azar, como se evidencia en el AUC cercano a 0.5 y en las métricas de ranking  bajas.


# 5 - Anexos y Referencias

- Codigo de DeepCTR Torch (con ejemplos) https://deepctr-torch.readthedocs.io/en/latest/Examples.html

- Chat con ChatGPT para intentar mejorar AUC: https://chatgpt.com/share/69002ca9-71c8-8006-a229-c2ddf54cfa8a

### Acá iran codigos que se intentaron immplementar pero que no funcionaron para mejorar el AUC y metricas

Evitar cold-start en evaluación Divide por usuario (leave-one-out por usuario, o al menos garantiza que todos los users de val/test aparecen en train).

In [ ]:
'''
def split_by_user(frame, user_col='user', random_state=42,
                  frac_train=0.8, frac_val=0.1, frac_test=0.1):
    rng = np.random.RandomState(random_state)
    parts_tr, parts_va, parts_te = [], [], []
    for u, g in frame.groupby(user_col):
        idx = np.arange(len(g))
        rng.shuffle(idx)
        n = len(idx)
        n_tr = max(1, int(n*frac_train))
        n_va = max(0, int(n*frac_val))
        n_te = max(1, n - n_tr - n_va)
        # Ajuste si sobra/falta
        while n_tr + n_va + n_te > n and n_va > 0:
            n_va -= 1
        tr = idx[:n_tr]; va = idx[n_tr:n_tr+n_va]; te = idx[n_tr+n_va:]
        parts_tr.append(g.iloc[tr])
        if len(va): parts_va.append(g.iloc[va])
        if len(te): parts_te.append(g.iloc[te])
    train_df = pd.concat(parts_tr).reset_index(drop=True)
    val_df   = pd.concat(parts_va).reset_index(drop=True) if parts_va else train_df.iloc[:0].copy()
    test_df  = pd.concat(parts_te).reset_index(drop=True) if parts_te else train_df.iloc[:0].copy()
    return train_df, val_df, test_df

train_df, val_df, test_df = split_by_user(df, user_col='user', random_state=seed)
'''

Revisar y corregir el desbalance de la etiqueta, mira el balance

In [59]:
print('Pos rate (train/val/test):',
      train_df['label'].mean(), val_df['label'].mean(), test_df['label'].mean())


Pos rate (train/val/test): 0.10539260577199822 0.10529621101109483 0.10548346588530766


Estan las features realmente

In [60]:
print('Features:', feature_names)
print({k: v.shape for k, v in X_train.items()})


Features: ['user', 'item', 'genre_enc', 'price']
{'user': (38219,), 'item': (38219,), 'genre_enc': (38219,), 'price': (38219,)}


In [61]:
print('pred_test mean/std:', pred_test.mean(), pred_test.std())


pred_test mean/std: 0.10831105432403147 0.09028884101629141


Hasta acá: Cold-start = 0 → no es por OOV.

Balance: ~26.6% positivos → razonable.

Predicciones: media 0.261, std 0.117 → no son constantes.

Con esto, si el modelo estuviera siendo evaluado correctamente, el AUC debería ser >0.50 (aunque sea poco). Cuando queda ≈0.50 en este escenario, casi siempre es por cómo se calcula el AUC o por desalineación, donde las features no entregar info relevante.

In [62]:
df.groupby('label')['price'].describe()
df['genre_enc'].nunique()
df.groupby('label')['genre_enc'].value_counts(normalize=True)


label  genre_enc
0      5            0.201900
       6            0.148576
       0            0.126044
       8            0.099815
       7            0.099020
       2            0.098388
       4            0.097639
       1            0.076464
       3            0.052154
1      5            0.198014
       6            0.147170
       0            0.133069
       7            0.105263
       8            0.103476
       2            0.096524
       4            0.093744
       1            0.072691
       3            0.050050
Name: proportion, dtype: float64

Los valores son muy parecidos entre label=0 y label=1, no hay información discriminante. Si genre_enc es casi constante (pocos géneros, mal codificados o mal mapeados) y price no se correlaciona con la preferencia, el modelo solo ve user e item.
Con pocas interacciones por usuario/item o etiquetas aleatorias (rating ≥ 4 → 1), el DeepFM no tiene cómo aprender diferencias reales.

Acá, con un codigo entregado por Copilot, se intento haceer una búsqueda pequeña de hiperparámetros + Early Stopping (selecciona mejor por val AUC)


In [ ]:
'''
from itertools import product
# Intenta usar callbacks de deepctr_torch si existen
try:
    from deepctr_torch.callbacks import EarlyStopping, ModelCheckpoint
    has_callbacks = True
except Exception:
    has_callbacks = False

# Helper: crea columnas de features con mismo embedding_dim para todas las SparseFeat

def make_feature_columns(emb_dim):
    cols_sparse = [
        SparseFeat('user', vocabulary_size=n_users, embedding_dim=emb_dim),
        SparseFeat('item', vocabulary_size=n_items, embedding_dim=emb_dim),
    ]
    if use_genre and 'genre_enc' in df.columns:
        cols_sparse.append(SparseFeat('genre_enc', vocabulary_size=n_genres, embedding_dim=emb_dim))
    cols_dense = []
    if use_price and 'price' in df.columns:
        cols_dense.append(DenseFeat('price', 1))
    return cols_sparse + cols_dense

# Helper: construye inputs respetando el orden de feature_names dado

def make_inputs(frame, names):
    data = {}
    for n in names:
        if n == 'price':
            data[n] = frame[n].astype('float32').values
        else:
            data[n] = frame[n].astype('int64').values
    return data

# Grid pequeña
emb_dims = [16, 32]
dropouts = [0.2, 0.4]
hidden_sets = [(128, 64), (256, 128)]
max_epochs = 15
patience = 3
batch_size_hp = 2048

best = {'auc': -1.0, 'conf': None, 'model': None, 'feature_names': None,
        'linear_cols': None, 'dnn_cols': None}

for emb_dim, ddrop, hset in product(emb_dims, dropouts, hidden_sets):
    lin_cols = make_feature_columns(emb_dim)
    dnn_cols = lin_cols
    names_hp = get_feature_names(lin_cols + dnn_cols)

    # Inputs para este modelo (pueden reutilizarse de los dataframes ya splitteados)
    Xtr = make_inputs(train_df, names_hp)
    Xva = make_inputs(val_df, names_hp) if len(val_df) > 0 else None
    ytr = train_df['label'].values
    yva = val_df['label'].values if len(val_df) > 0 else None

    m = DeepFM(
        linear_feature_columns=lin_cols,
        dnn_feature_columns=dnn_cols,
        task='binary',
        device=device,
        dnn_hidden_units=hset,
        dnn_dropout=ddrop,
    )
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['auc'])

    cbs = []
    if has_callbacks and len(val_df) > 0:
        try:
            es = EarlyStopping(monitor='val_auc', patience=patience, mode='max', verbose=1)
            cbs.append(es)
        except Exception:
            pass
    try:
        m.fit(Xtr, ytr,
              batch_size=batch_size_hp, epochs=max_epochs, verbose=2,
              validation_data=(Xva, yva) if len(val_df) > 0 else None,
              shuffle=True,
              callbacks=cbs if cbs else None)
    except TypeError:
        # Si la versión no soporta callbacks, entrenar sin ellos
        m.fit(Xtr, ytr,
              batch_size=batch_size_hp, epochs=max_epochs, verbose=2,
              validation_data=(Xva, yva) if len(val_df) > 0 else None,
              shuffle=True)

    # Val AUC para selección del mejor
    if len(val_df) > 0:
        pred_va = m.predict(Xva, batch_size=4096)
        pred_va = np.asarray(pred_va).reshape(-1)
        val_auc = roc_auc_score(yva, pred_va)
    else:
        # Si no hay val, usa AUC de train (no ideal, pero evita bloqueo)
        pred_tr = m.predict(Xtr, batch_size=4096)
        pred_tr = np.asarray(pred_tr).reshape(-1)
        val_auc = roc_auc_score(ytr, pred_tr)

    print(f'[HP] emb_dim={emb_dim}, dropout={ddrop}, hidden={hset} -> val_auc={val_auc:.4f}')
    if val_auc > best['auc']:
        best.update({'auc': val_auc, 'conf': (emb_dim, ddrop, hset), 'model': m,
                     'feature_names': names_hp, 'linear_cols': lin_cols, 'dnn_cols': dnn_cols})

# Fija como modelo activo el mejor encontrado
assert best['model'] is not None
model = best['model']
feature_names = best['feature_names']
linear_feature_columns = best['linear_cols']
dnn_feature_columns = best['dnn_cols']
print('Mejor config -> emb_dim=%s, dropout=%s, hidden=%s | val_auc=%.4f' % (*best['conf'], best['auc']))
'''

cpu
Train on 31552 samples, validate on 3944 samples, 16 steps per epoch
Epoch 1/15
0s - loss:  0.6399 - auc:  0.5148 - val_auc:  0.5134
Epoch 1/15
0s - loss:  0.6399 - auc:  0.5148 - val_auc:  0.5134
Epoch 2/15
0s - loss:  0.5241 - auc:  0.5533 - val_auc:  0.5150
Epoch 2/15
0s - loss:  0.5241 - auc:  0.5533 - val_auc:  0.5150
Epoch 3/15
0s - loss:  0.3656 - auc:  0.5797 - val_auc:  0.5164
Epoch 3/15
0s - loss:  0.3656 - auc:  0.5797 - val_auc:  0.5164
Epoch 4/15
0s - loss:  0.3017 - auc:  0.6546 - val_auc:  0.5103
Epoch 4/15
0s - loss:  0.3017 - auc:  0.6546 - val_auc:  0.5103
Epoch 5/15
0s - loss:  0.2952 - auc:  0.7345 - val_auc:  0.5080
Epoch 5/15
0s - loss:  0.2952 - auc:  0.7345 - val_auc:  0.5080
Epoch 6/15
0s - loss:  0.2864 - auc:  0.7707 - val_auc:  0.5095
Epoch 00006: early stopping
[HP] emb_dim=16, dropout=0.2, hidden=(128, 64) -> val_auc=0.5095
cpu
Train on 31552 samples, validate on 3944 samples, 16 steps per epoch
Epoch 6/15
0s - loss:  0.2864 - auc:  0.7707 - val_auc:  